# 11 · Final results

All numbers, figures and example outputs for the final report, pulled straight from `group26/artifacts`.
Run on CARC from `group26/notebooks` with the `mfr-dpo` env. Read-only except the output folder (`OUT`).

**What the numbers mean**
- **Accuracy**: % of preference pairs where the trained model prefers the chosen answer more than the base model does (margin > 0).
- **Retention**: change in accuracy on each earlier behavior, from right after it was learned to the end of training, averaged over the earlier behaviors. 0 = nothing forgotten, negative = forgetting. Same formula as `summary_table` in `src/mfr_analysis.py`.
- **New task**: accuracy on the last behavior trained, at the end.
- **Final avg**: mean accuracy over Helpful, Safety and Quality at the end.
- **Cell**: one training order × seed. Locked test = 12 cells (orders 1–4 × seeds 0–2). Validation = 8 cells (seeds 0–1).
- **Paired difference**: A − B in the same cell, averaged over cells. `[lo, hi]` is a 95% bootstrap interval over cells, `k/n` is how many cells A beats B.

## 0. Setup

In [ ]:
import os, re, json, glob, html, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from IPython.display import display, HTML, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)

ROOT = Path(os.environ.get('G26', '/project2/xiangren_1987/group26'))
ART = ROOT / 'artifacts'
RUNS, JOINT_RUNS = ART / 'runs', ART / 'joint_runs'
PID_RUNS = ART / 'pid_gain_search_v1' / 'runs'          # Jesse's runs, with his own controls
DATA = ROOT / 'data' / 'v2'
OUT = Path(os.environ.get('G26_OUT', ART / 'analysis' / 'final_results'))
OUT.mkdir(parents=True, exist_ok=True)

MAIN = ['none', 'random', 'random_high', 'mfr', 'lowest_margin', 'dapr_weak']
VARIANTS = ['none', 'random', 'random_high', 'mfr', 'mfr_balanced', 'mfr_at_risk', 'pid',
            'lowest_margin', 'dapr', 'dapr_weak']
PID_RUN_NAME = 'pid_i000_d100'   # Jesse's final setting (Kp 1, Ki 0, Kd 1), called 'pid' below
LABEL = {'none': 'No replay', 'random': 'Random 10%', 'random_high': 'Random 14.3%', 'mfr': 'MFR',
         'mfr_balanced': 'Balanced MFR', 'mfr_at_risk': 'At-Risk MFR', 'pid': 'Trend-aware MFR',
         'lowest_margin': 'Lowest Margin', 'dapr': 'DAPR (strong)', 'dapr_weak': 'DAPR-weak',
         'joint': 'Joint training', 'base': 'Base model'}
COLOR = {'none': '#8a8984', 'random': '#2a78d6', 'random_high': '#86b6ef', 'mfr': '#eb6834',
         'mfr_balanced': '#eda100', 'mfr_at_risk': '#e87ba4', 'pid': '#b8461b', 'lowest_margin': '#1baf7a',
         'dapr': '#a99ce8', 'dapr_weak': '#6250d6', 'joint': '#0b0b0b', 'base': '#c9c8c3'}
BEH = ['helpful', 'safe', 'quality']
BLABEL = {'helpful': 'Helpful', 'safe': 'Safety', 'quality': 'Quality'}
BCOLOR = {'helpful': '#2a78d6', 'safe': '#eb6834', 'quality': '#1baf7a'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e5e4e0'
FULL_W = 6.5

def style():
    names = {f.name for f in fm.fontManager.ttflist}
    family = next((n for n in ['Calibri', 'Carlito'] if n in names), 'DejaVu Sans')
    plt.rcParams.update({
        'font.family': family, 'font.size': 9.5, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
        'axes.titlelocation': 'left', 'axes.labelsize': 9.5, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
        'legend.fontsize': 8.5, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.edgecolor': '#8a8984', 'axes.linewidth': 0.8, 'figure.dpi': 130, 'savefig.dpi': 300,
        'figure.constrained_layout.use': True, 'pdf.fonttype': 42, 'figure.facecolor': 'white'})
style()

TABLES, FIGS = {}, {}
def show(df, name, caption=None, decimals=2):
    TABLES[name] = df
    if caption:
        display(Markdown(f'**{name}.** {caption}'))
    display(df.round(decimals))

def keep(fig, name):
    FIGS[name] = fig
    display(fig)
    plt.close(fig)

def lab(methods):
    return [LABEL.get(m, m) for m in methods]

def grid(ax, axis='x'):
    ax.grid(axis=axis, color=GRID, lw=0.7)
    ax.set_axisbelow(True)

def boot_ci(values, n=5000, seed=0):
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan, np.nan
    bs = np.random.default_rng(seed).choice(v, (n, len(v))).mean(axis=1)
    return np.percentile(bs, 2.5), np.percentile(bs, 97.5)

print('reading from', ART)
print('saving to   ', OUT)

## 1. Load everything
Each results file has one accuracy per (stage, behavior). `cell_metrics` turns one run into one row: retention, new task, final avg and the per-behavior numbers. It also keeps the forgetting curve (accuracy change on an earlier behavior after each later stage).

In [ ]:
def run_dir(f):
    f = Path(f)
    return f.parents[1] if f.parent.name == 'locked_test' else f.parent

def cell_metrics(path):
    g = pd.read_csv(path)
    g = g[g['stage'] >= 1]
    last = int(g['stage'].max())
    learned = g.drop_duplicates('stage').set_index('trained_on')['stage'].to_dict()
    acc = lambda s, b: float(g.loc[(g['stage'] == s) & (g['eval_set'] == b), 'accuracy'].iloc[0])
    evals = [b for b in BEH if b in set(g['eval_set'])]
    r = {'run': g['run_name'].iloc[0], 'method': str(g['method'].iloc[0]), 'order': int(g['order_id'].iloc[0]),
         'seed': int(g['seed'].iloc[0]), 'final_avg': np.mean([acc(last, b) for b in evals])}
    for b in evals:
        r[f'final_{b}'] = acc(last, b)
    early = [b for b, s in learned.items() if s < last and b in evals]
    curve = []
    if early:
        r['new_task'] = acc(last, g.loc[g['stage'] == last, 'trained_on'].iloc[0])
        change = {b: acc(last, b) - acc(learned[b], b) for b in early}
        r['retention'] = np.mean(list(change.values()))
        for b, v in change.items():
            r[f'ret_{b}'] = v
        for b in early:
            for s in range(learned[b], last + 1):
                curve.append({'method': r['method'], 'order': r['order'], 'seed': r['seed'], 'behavior': b,
                              'learned_at': learned[b], 'stages_later': s - learned[b],
                              'change': acc(s, b) - acc(learned[b], b)})
    return r, curve

def load_cells(pattern, need_complete=True):
    rows, curves, bad = [], [], []
    for f in sorted(glob.glob(str(pattern))):
        if need_complete and not (run_dir(f) / 'COMPLETE.json').exists():
            continue
        try:
            r, c = cell_metrics(f)
            rows.append(r)
            curves += c
        except Exception as e:
            bad.append(f'{f}: {e}')
    if bad:
        print(f'{len(bad)} files skipped, first: {bad[0]}')
    return pd.DataFrame(rows), pd.DataFrame(curves)

TEST, TEST_CURVE = load_cells(RUNS / 'v2_o[1-4]_*_s[0-2]' / 'locked_test' / 'results_test.csv')
VAL, VAL_CURVE = load_cells(RUNS / 'v2_o[1-4]_*_s[01]' / 'results.csv')
JVAL, _ = load_cells(PID_RUNS / 'v2_o[1-4]_*_s[01]' / 'results.csv')
JVAL['method'] = JVAL['method'].replace({PID_RUN_NAME: 'pid'})
JVAL = JVAL[JVAL['method'].isin(['none', 'random', 'mfr', 'lowest_margin', 'pid'])]
JOINT, _ = load_cells(JOINT_RUNS / 'v2_joint_s*' / 'locked_test' / 'results_test.csv')

inv = pd.concat([TEST.assign(source='locked test'), VAL.assign(source='validation'),
                 JVAL.assign(source="validation (Jesse's folder)")]).groupby(['method', 'source']).size().unstack(fill_value=0)
inv = inv.reindex([m for m in VARIANTS if m in inv.index] + [m for m in inv.index if m not in VARIANTS])
inv.index = lab(inv.index)
inv.loc['Joint training'] = 0
inv.loc['Joint training', 'locked test'] = len(JOINT)
show(inv, 'T0_inventory', 'Completed runs found per method.', 0)

## 2. Main results (locked test)
The six main methods, 12 cells each, plus joint training (all data at once, no sequence) as a reference for the best you could hope for.

In [ ]:
COLS = {'retention': 'Retention', 'new_task': 'New task', 'final_avg': 'Final avg', 'final_helpful': 'Final Helpful',
        'final_safe': 'Final Safety', 'final_quality': 'Final Quality', 'ret_helpful': 'Ret. Helpful',
        'ret_safe': 'Ret. Safety', 'ret_quality': 'Ret. Quality'}

def method_table(df, methods, cols):
    present = [m for m in methods if m in set(df['method'])]
    t = df.groupby('method')[cols].mean().reindex(present)
    t.insert(0, 'cells', df.groupby('method').size().reindex(present))
    t.index = lab(t.index)
    return t.rename(columns=COLS)

t1 = method_table(TEST, MAIN, list(COLS))
t1.loc['Joint training', ['cells', 'Final avg', 'Final Helpful', 'Final Safety', 'Final Quality']] = [
    len(JOINT), *JOINT[['final_avg', 'final_helpful', 'final_safe', 'final_quality']].mean()]
show(t1, 'T1_locked_test',
     'Locked test, mean over cells. Ret. columns are retention for one behavior, averaged over the orders where '
     'it is an earlier behavior (Helpful: orders 1–3, Safety: 1, 2, 4, Quality: 3, 4). Joint has no sequence, so no retention.')

In [ ]:
def paired(df, a, b, col):
    x = df[df['method'] == a].set_index(['order', 'seed'])[col]
    y = df[df['method'] == b].set_index(['order', 'seed'])[col]
    d = (x - y).dropna()
    lo, hi = boot_ci(d)
    return {'diff': d.mean(), 'lo': lo, 'hi': hi, 'better': int((d > 0).sum()), 'cells': len(d)}

def paired_table(df, pairs, cols=('retention', 'new_task', 'final_avg')):
    rows = [{'comparison': f'{LABEL[a]} − {LABEL[b]}', 'a': a, 'b': b, 'metric': c, **paired(df, a, b, c)}
            for a, b in pairs for c in cols]
    return pd.DataFrame(rows)

def fmt_pairs(t):
    t = t.assign(text=t.apply(lambda r: f"{r['diff']:+.2f} [{r['lo']:+.2f}, {r['hi']:+.2f}]  {r['better']}/{r['cells']}", axis=1))
    out = t.pivot(index='comparison', columns='metric', values='text')
    out = out.reindex(list(dict.fromkeys(t['comparison'])))
    return out[[c for c in ['retention', 'new_task', 'final_avg'] if c in out]].rename(columns=COLS)

PAIRS = [('random', 'none'), ('random_high', 'random'), ('mfr', 'random'), ('lowest_margin', 'random'),
         ('lowest_margin', 'mfr'), ('dapr_weak', 'random'), ('dapr_weak', 'lowest_margin'), ('dapr_weak', 'none')]
T2 = paired_table(TEST, PAIRS)
TABLES['T2_paired_locked_test_raw'] = T2
show(fmt_pairs(T2), 'T2_paired_locked_test',
     'Paired differences on the locked test: mean [95% CI] and cells where the first method is higher. '
     'An interval that does not cross 0 is a reliable difference.')

In [ ]:
# Figure 1: forgetting per method, dots are the 12 cells
fig, ax = plt.subplots(figsize=(FULL_W, 2.7))
for i, m in enumerate(MAIN):
    v = TEST.loc[TEST['method'] == m, 'retention']
    y = len(MAIN) - 1 - i
    lo, hi = boot_ci(v)
    ax.barh(y, v.mean(), color=COLOR[m], height=0.62, zorder=2)
    ax.errorbar(v.mean(), y, xerr=[[v.mean() - lo], [hi - v.mean()]], color=INK, lw=1, capsize=2.5, zorder=4)
    ax.scatter(v, y + np.random.default_rng(i).uniform(-0.2, 0.2, len(v)), s=9, color=INK, alpha=0.35, lw=0, zorder=3)
    ax.text(1.01, y, f'{v.mean():.2f}', va='center', ha='left', fontsize=8.5, transform=ax.get_yaxis_transform())
ax.axvline(0, color=INK, lw=0.8)
ax.set_yticks(range(len(MAIN)), lab(MAIN[::-1]))
ax.tick_params(axis='y', length=0)
ax.set_xlabel('Retention (accuracy points, 0 = no forgetting)')
ax.set_title('Forgetting of earlier behaviors, locked test')
grid(ax)
keep(fig, 'F1_retention_locked_test')

In [ ]:
# Figure 2: which behavior is forgotten
fig, ax = plt.subplots(figsize=(FULL_W, 2.7))
w = 0.8 / len(MAIN)
for j, m in enumerate(MAIN):
    vals = [TEST.loc[TEST['method'] == m, f'ret_{b}'].mean() for b in BEH]
    ax.bar(np.arange(3) + (j - (len(MAIN) - 1) / 2) * w, vals, w * 0.92, color=COLOR[m], label=LABEL[m], zorder=2)
ax.axhline(0, color=INK, lw=0.8)
ax.set_xticks(range(3), [BLABEL[b] for b in BEH])
ax.set_ylabel('Retention (points)')
ax.set_title('Forgetting by behavior, locked test')
grid(ax, 'y')
fig.legend(*ax.get_legend_handles_labels(), loc='outside upper center', ncol=3)
keep(fig, 'F2_retention_by_behavior')

In [ ]:
# Figure 3: forgetting curve of the first behavior learned, after each later stage
c = TEST_CURVE[TEST_CURVE['learned_at'] == 1].groupby(['method', 'stages_later'])['change'].mean().unstack()
fig, ax = plt.subplots(figsize=(FULL_W * 0.6, 2.6))
for m in MAIN:
    if m in c.index:
        ax.plot(c.columns, c.loc[m], marker='o', ms=4, lw=1.8, color=COLOR[m], label=LABEL[m])
ax.axhline(0, color=INK, lw=0.8)
ax.set_xticks([0, 1, 2], ['just learned', '+1 stage', '+2 stages'])
ax.set_ylabel('Accuracy change (points)')
ax.set_title('First behavior, as later stages are trained')
grid(ax, 'y')
ax.legend(loc='lower left', fontsize=7.5)
keep(fig, 'F3_forgetting_curve')

In [ ]:
# Figure 4: paired differences with 95% intervals
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.8), sharey=True)
comps = list(dict.fromkeys(T2['comparison']))
for ax, metric in zip(axes, ['retention', 'final_avg']):
    t = T2[T2['metric'] == metric].set_index('comparison').reindex(comps)
    y = np.arange(len(t))[::-1]
    col = [COLOR[a] for a in t['a']]
    ax.errorbar(t['diff'], y, xerr=[t['diff'] - t['lo'], t['hi'] - t['diff']], fmt='none', ecolor=INK2, lw=1, capsize=2)
    ax.scatter(t['diff'], y, color=col, s=28, zorder=3)
    ax.axvline(0, color=INK, lw=0.8)
    ax.set_title(COLS[metric] + ' difference')
    grid(ax)
axes[0].set_yticks(np.arange(len(comps))[::-1], comps)
axes[0].tick_params(axis='y', length=0)
fig.supxlabel('Points (first method − second), 95% bootstrap interval over cells', fontsize=8.5)
keep(fig, 'F4_paired_differences')

In [ ]:
# Figure 5: forgetting vs final average, with joint training as the reference line
s = TEST.groupby('method')[['retention', 'final_avg']].mean().reindex(MAIN)
fig, ax = plt.subplots(figsize=(FULL_W * 0.6, 2.8))
ax.axhline(JOINT['final_avg'].mean(), color=INK, lw=1, ls='--')
ax.text(1.0, JOINT['final_avg'].mean(), 'Joint training ', fontsize=8, color=INK, ha='right', va='bottom',
        transform=ax.get_yaxis_transform())
for m, r in s.iterrows():
    ax.scatter(r['retention'], r['final_avg'], s=45, color=COLOR[m], zorder=3)
    ax.annotate(LABEL[m], (r['retention'], r['final_avg']), xytext=(5, -3), textcoords='offset points', fontsize=8)
ax.set_xlabel('Retention (points)')
ax.set_ylabel('Final avg accuracy (%)')
ax.set_title('Less forgetting, better final model')
grid(ax, 'both')
keep(fig, 'F5_retention_vs_final')

## 3. MFR variants (validation, 8 cells)
Each variant changes one thing about MFR to test one idea. Trend-aware MFR is Jesse's PID selector with the history term off (Kp 1, Ki 0, Kd 1). It was trained in his own folder with his own controls, so its paired numbers compare against **his** MFR and Lowest Margin runs.

In [ ]:
VCOLS = ['retention', 'new_task', 'final_avg']
ours = [m for m in VARIANTS if m != 'pid']
t3 = method_table(VAL, ours, VCOLS)
for b, col in [('mfr', 'vs MFR'), ('lowest_margin', 'vs Lowest Margin')]:
    t3[col] = [paired(VAL, m, b, 'retention')['diff'] if m != b else np.nan for m in ours if m in set(VAL['method'])]
pid = JVAL[JVAL['method'] == 'pid']
if len(pid):
    row = {'cells': len(pid), **pid[VCOLS].mean().rename(COLS).to_dict(),
           'vs MFR': paired(JVAL, 'pid', 'mfr', 'retention')['diff'],
           'vs Lowest Margin': paired(JVAL, 'pid', 'lowest_margin', 'retention')['diff']}
    t3.loc[LABEL['pid'] + ' *'] = pd.Series(row)
show(t3, 'T3_variants_validation',
     'Validation, mean over cells. "vs" columns are paired retention differences (positive = forgets less). '
     '* paired against Jesse\'s own controls.')

check = []
for m in ['none', 'random', 'mfr', 'lowest_margin']:
    d = (JVAL[JVAL['method'] == m].set_index(['order', 'seed'])[VCOLS] -
         VAL[VAL['method'] == m].set_index(['order', 'seed'])[VCOLS]).dropna()
    if len(d):
        check.append({'method': LABEL[m], 'cells': len(d), **{f'mean diff {COLS[c]}': d[c].mean() for c in VCOLS},
                      'max |diff| retention': d['retention'].abs().max()})
show(pd.DataFrame(check).set_index('method'), 'T3b_jesse_controls_check',
     "Jesse's control runs minus ours on the same cells. Small mean differences = his code reproduces ours; "
     'the max column shows rerun noise for a single cell.')

In [ ]:
# Figure 6: variants on validation
order_v = [m for m in VARIANTS if m in set(VAL['method']) or (m == 'pid' and len(pid))]
fig, ax = plt.subplots(figsize=(FULL_W, 3.0))
for i, m in enumerate(order_v):
    src = JVAL if m == 'pid' else VAL
    v = src.loc[src['method'] == m, 'retention']
    y = len(order_v) - 1 - i
    lo, hi = boot_ci(v)
    ax.barh(y, v.mean(), color=COLOR[m], height=0.62, zorder=2, hatch='///' if m == 'pid' else None,
            edgecolor='white' if m == 'pid' else None)
    ax.errorbar(v.mean(), y, xerr=[[v.mean() - lo], [hi - v.mean()]], color=INK, lw=1, capsize=2.5, zorder=4)
    ax.text(1.01, y, f'{v.mean():.2f}', va='center', fontsize=8.5, transform=ax.get_yaxis_transform())
ax.axvline(0, color=INK, lw=0.8)
ax.set_yticks(range(len(order_v)), lab(order_v[::-1]))
ax.tick_params(axis='y', length=0)
ax.set_xlabel('Retention (points, validation)')
ax.set_title('MFR variants and DAPR, validation (hatched: Jesse\'s runs)')
grid(ax)
keep(fig, 'F6_variants_validation')

## 4. What gets replayed
From each run's `replay_log.csv` (stages 2 and 3, validation runs). Every method gets 222 replay slots per stage (Random 14.3% gets 333).
- **Distinct pairs**: how many different stored pairs fill those slots. Low = the same pairs are replayed again and again.
- **Top-20 share**: % of slots taken by the 20 most-replayed pairs.
- **Overlap**: for two methods in the same order, seed and stage, the share of distinct replayed pairs they have in common (Jaccard).

In [ ]:
def replay_logs(root, methods, rename=None):
    out = {}
    for f in glob.glob(str(root / 'v2_o[1-4]_*_s[01]' / 'stage[23]_*' / 'replay_log.csv')):
        run = Path(f).parents[1].name
        mm = re.match(r'v2_o(\d)_(.+)_s(\d)$', run)
        m = (rename or {}).get(mm.group(2), mm.group(2))
        if m in methods and m != 'none':
            d = pd.read_csv(f)
            if len(d):
                out[(m, int(mm.group(1)), int(mm.group(3)), Path(f).parent.name)] = d
    return out

LOGS = replay_logs(RUNS, set(VARIANTS))
JLOGS = replay_logs(PID_RUNS, {'pid', 'mfr', 'lowest_margin', 'random'}, {PID_RUN_NAME: 'pid'})

rows = []
for (m, o, s, st), d in list(LOGS.items()) + [(k, v) for k, v in JLOGS.items() if k[0] == 'pid']:
    c = d.groupby(['dataset', 'id']).size().sort_values(ascending=False)
    rows.append({'method': m, 'slots': len(d), 'distinct': len(c), 'max_repeat': c.max(),
                 'top20': 100 * c.iloc[:20].sum() / len(d), 'once': 100 * (c == 1).mean()})
R = pd.DataFrame(rows)
t4 = R.groupby('method').agg(stages=('slots', 'size'), slots=('slots', 'mean'), distinct=('distinct', 'mean'),
                             max_repeat=('max_repeat', 'mean'), top20=('top20', 'mean'), once=('once', 'mean'))
t4 = t4.reindex([m for m in VARIANTS if m in t4.index])
t4.index = lab(t4.index)
show(t4.rename(columns={'distinct': 'Distinct pairs', 'max_repeat': 'Most replays of one pair',
                        'top20': 'Top-20 share (%)', 'once': 'Replayed once (%)'}),
     'T4_replay_concentration', 'Replay concentration per stage, mean over 16 stages (8 cells × 2 stages).', 1)

def overlap(logs, a, b):
    jac = []
    for (m, o, s, st), d in logs.items():
        if m == a and (b, o, s, st) in logs:
            A = set(map(tuple, d[['dataset', 'id']].values))
            B = set(map(tuple, logs[(b, o, s, st)][['dataset', 'id']].values))
            jac.append(len(A & B) / len(A | B))
    return np.mean(jac) if jac else np.nan, len(jac)

rows = []
for src, logs, pairs in [('ours', LOGS, [('mfr', 'lowest_margin'), ('random', 'lowest_margin'), ('mfr', 'random'),
                                         ('dapr_weak', 'lowest_margin')]),
                         ("Jesse's folder", JLOGS, [('pid', 'lowest_margin'), ('pid', 'mfr')])]:
    for a, b in pairs:
        j, n = overlap(logs, a, b)
        rows.append({'pair': f'{LABEL[a]} vs {LABEL[b]}', 'runs from': src, 'stages': n, 'Jaccard (%)': 100 * j})
show(pd.DataFrame(rows).set_index('pair'), 'T4b_replay_overlap',
     'How many of the same pairs two methods replay (100% = identical sets).', 1)

In [ ]:
# Figure 7: replay concentration
m7 = [m for m in VARIANTS if m in set(R['method'])]
g = R.groupby('method')[['distinct', 'top20']].mean().reindex(m7)
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.9), sharey=True)
y = np.arange(len(m7))[::-1]
for ax, col, title in [(axes[0], 'distinct', 'Distinct pairs replayed per stage'),
                       (axes[1], 'top20', 'Share of slots for the top-20 pairs (%)')]:
    ax.barh(y, g[col], color=[COLOR[m] for m in m7], height=0.62, zorder=2)
    for yi, v in zip(y, g[col]):
        ax.text(v + 1, yi, f'{v:.0f}', va='center', fontsize=8)
    ax.set_title(title)
    grid(ax)
axes[0].set_yticks(y, lab(m7))
axes[0].tick_params(axis='y', length=0)
keep(fig, 'F7_replay_concentration')

## 5. Which pairs are forgotten (validation, pair level)
Uses the per-pair margins saved after every stage. A pair is **learned** if its margin is above 0 right after its behavior was trained, and **forgotten** if it is learned but at or below 0 at the end.
- **Shared core**: pairs that all five midterm methods learned and all five forgot (the midterm's 216).
- **Length**: whether the chosen answer is shorter than the rejected one.

In [ ]:
PAIR_METHODS = ['none', 'random', 'random_high', 'mfr', 'lowest_margin', 'dapr_weak']
CORE_METHODS = ['none', 'random', 'random_high', 'mfr', 'lowest_margin']   # the midterm's five

def stage_dir(rd, s, k):
    ds = s['order'][k - 1]
    p = rd / f'stage{k}_{ds}'
    if p.exists() or k != 1 or not s.get('stage1_from'):
        return p
    src = Path(s['stage1_from'])
    return (src if src.exists() else rd.parent / src.name) / f'stage1_{ds}'

def margins(rd, s, k, ds):
    return pd.read_csv(stage_dir(rd, s, k) / f'margins_{ds}_val.csv').set_index('id')['margin']

frames = []
for f in sorted(RUNS.glob('v2_o[1-4]_*_s[01]/settings.json')):
    rd = f.parent
    s = json.loads(f.read_text())
    if s.get('method') not in PAIR_METHODS or not (rd / 'COMPLETE.json').exists():
        continue
    order, last = s['order'], len(s['order'])
    for k, ds in enumerate(order[:-1], 1):
        try:
            a, z = margins(rd, s, k, ds).align(margins(rd, s, last, ds), join='inner')
        except FileNotFoundError as e:
            print('missing margins:', e)
            continue
        frames.append(pd.DataFrame({'method': s['method'], 'order': int(s['order_id']), 'seed': int(s['seed']),
                                    'behavior': ds, 'id': a.index, 'learned': a.values, 'final': z.values}))
PAIRS_DF = pd.concat(frames, ignore_index=True)
PAIRS_DF['forgotten'] = (PAIRS_DF['learned'] > 0) & (PAIRS_DF['final'] <= 0)

wide_l = PAIRS_DF.pivot_table(index=['order', 'seed', 'behavior', 'id'], columns='method', values='learned')
wide_f = PAIRS_DF.pivot_table(index=['order', 'seed', 'behavior', 'id'], columns='method', values='final')
cm = [m for m in CORE_METHODS if m in wide_l]
known_all = (wide_l[cm] > 0).all(axis=1)
CORE = known_all & (wide_f[cm] <= 0).all(axis=1)
core_ids = CORE[CORE].index
dapr_keeps = int(((wide_l.loc[core_ids, 'dapr_weak'] > 0) & (wide_f.loc[core_ids, 'dapr_weak'] > 0)).sum()) \
    if 'dapr_weak' in wide_l else np.nan

rows = []
for m in PAIR_METHODS:
    g = PAIRS_DF[PAIRS_DF['method'] == m]
    if g.empty:
        continue
    f_ = g['forgotten']
    in_core = g.set_index(['order', 'seed', 'behavior', 'id']).index.isin(core_ids)
    rows.append({'method': LABEL[m], 'learned pairs': int((g['learned'] > 0).sum()), 'forgotten': int(f_.sum()),
                 'forgotten (% of learned)': 100 * f_.sum() / max((g['learned'] > 0).sum(), 1),
                 'in shared core': int((f_ & in_core).sum())})
show(pd.DataFrame(rows).set_index('method'), 'T5_forgotten_pairs',
     f'Validation pairs summed over 8 cells. Shared core = {len(core_ids)} pairs forgotten by all five midterm methods; '
     f'DAPR-weak keeps {dapr_keeps} of them.', 1)

In [ ]:
def read_jsonl(p):
    with open(p) as fh:
        return [json.loads(line) for line in fh if line.strip()]

def n_tokens(r, field):
    v = r.get(f'{field}_tokens')
    return float(v) if v is not None else float(len(str(r.get(field, '')).split()))

TEXT = {}
for b in BEH:
    for split in ['val', 'test']:
        p = DATA / f'{b}_{split}.jsonl'
        if p.exists():
            for r in read_jsonl(p):
                TEXT[r['id']] = r
gap = pd.Series({i: n_tokens(r, 'chosen') - n_tokens(r, 'rejected') for i, r in TEXT.items()})

L = PAIRS_DF[PAIRS_DF['learned'] > 0].copy()
L['chosen'] = np.where(gap.reindex(L['id']).values < 0, 'shorter', 'longer or equal')
t6 = (L.groupby(['method', 'chosen'])['forgotten'].mean().mul(100).unstack().reindex(PAIR_METHODS))
t6.index = lab(t6.index)
share = {BLABEL[b]: 100 * (gap[[i for i in gap.index if i.startswith(b) and '-val-' in i]] < 0).mean() for b in BEH}
show(t6, 'T6_length_effect',
     'Forgotten (% of learned pairs), split by whether the chosen answer is shorter than the rejected one. '
     'Validation pairs where chosen is shorter: ' + ', '.join(f'{k} {v:.0f}%' for k, v in share.items()) + '.', 1)

fig, ax = plt.subplots(figsize=(FULL_W * 0.6, 2.6))
x = np.arange(len(t6))
for dx, col, c in [(-0.2, 'shorter', '#e34948'), (0.2, 'longer or equal', '#8fa8c8')]:
    ax.bar(x + dx, t6[col], 0.38, color=c, label=f'chosen answer {col}', zorder=2)
ax.set_xticks(x, t6.index, rotation=30, ha='right', rotation_mode='anchor')
ax.set_ylabel('Forgotten (% of learned)')
ax.set_title('Pairs that prefer the shorter answer are lost more')
grid(ax, 'y')
ax.legend(fontsize=7.5)
keep(fig, 'F8_length_effect')

## 6. Generated answers: safety and instruction following
Final checkpoints, greedy decoding. WildGuard labels each answer.
- **HarmBench harmful %**: share of answers to harmful requests that are harmful (lower is safer).
- **XSTest over-refusal %**: share of safe-but-scary-sounding requests the model refuses (lower is better).
- **PKU harmful %**: harmful answers on the PKU safety test prompts. **After Safe → end** is how much it rises from right after the Safety stage to the end (forgetting in generations).
- **IFEval**: % of prompts where every formatting instruction is followed (prompt-level strict).

In [ ]:
def safety_summary(path):
    d = pd.read_csv(path).set_index('suite')
    r = {}
    for suite in d.index:
        for col, short in [('harm_rate_pct', 'harm'), ('refusal_rate_pct', 'refusal'), ('over_refusal_rate_pct', 'over_refusal')]:
            if col in d.columns and pd.notna(d.loc[suite, col]):
                r[f'{suite}_{short}'] = float(d.loc[suite, col])
    return r

def method_of(run):
    mm = re.match(r'v2_o\d_(.+)_s\d$', run)
    return mm.group(1) if mm else run

def ifeval_score(f):
    res = json.load(open(f))['results']
    t = res.get('ifeval', next(iter(res.values())))
    get = lambda k: next((v for kk, v in t.items() if kk.startswith(k) and 'stderr' not in kk), np.nan)
    return 100 * get('prompt_level_strict_acc'), 100 * get('inst_level_strict_acc')

rows = []
for f in glob.glob(str(RUNS / 'v2_o[1-4]_*_s[0-2]' / 'generation' / 'final_eval' / 'safety' / 'summary.csv')):
    run = Path(f).parents[3].name
    r = {'run': run, 'method': method_of(run), **safety_summary(f)}
    post = RUNS / run / 'generation' / 'post_stage_safe' / 'safety' / 'summary.csv'
    if post.exists():
        r['safe_after_stage'] = safety_summary(post).get('safe_test_harm', np.nan)
    ife = sorted((RUNS / run / 'generation' / 'final_eval' / 'ifeval').glob('*/results_*.json'))
    if ife:
        r['ifeval_prompt'], r['ifeval_inst'] = ifeval_score(ife[-1])
    rows.append(r)
GEN = pd.DataFrame(rows)
GEN['safe_increase'] = GEN.get('safe_test_harm') - GEN.get('safe_after_stage', np.nan)

base = {}
for f in glob.glob(str(ART / 'base_model' / '**' / 'safety' / 'summary.csv'), recursive=True):
    base.update(safety_summary(f))
for f in sorted(glob.glob(str(ART / 'base_model' / '**' / 'results_*.json'), recursive=True)):
    if 'ifeval' in f and 'smoke' not in f:
        base['ifeval_prompt'], base['ifeval_inst'] = ifeval_score(f)

GCOLS = {'harmbench_harm': 'HarmBench harmful %', 'harmbench_refusal': 'HarmBench refusal %',
         'xstest_over_refusal': 'XSTest over-refusal %', 'safe_test_harm': 'PKU harmful %',
         'safe_increase': 'PKU harmful: after Safe → end', 'ifeval_prompt': 'IFEval prompt %', 'ifeval_inst': 'IFEval instruction %'}
gm = [m for m in ['none', 'random', 'random_high', 'mfr', 'lowest_margin', 'dapr_weak', 'dapr'] if m in set(GEN['method'])]
t7 = GEN.groupby('method')[[c for c in GCOLS if c in GEN]].mean().reindex(gm)
t7.insert(0, 'runs', GEN.groupby('method').size().reindex(gm))
t7.loc['base'] = pd.Series({'runs': 1, **{k: v for k, v in base.items() if k in t7.columns}})
t7.index = lab(t7.index)
show(t7.rename(columns=GCOLS), 'T7_generation',
     'Mean over runs (12 per main method, 8 for DAPR strong). PKU rates sit near 0, so the PKU columns are only directional. '
     'IFEval gaps of about 1 point are within noise.')

In [ ]:
# Figure 9: safety trade-off and IFEval
fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.9), width_ratios=[1.15, 1])
ax = axes[0]
pts = GEN.groupby('method')[['harmbench_harm', 'xstest_over_refusal']].mean().reindex(gm)
placed = []
for m, r in pts.sort_values('harmbench_harm').iterrows():
    x0, y0 = r['xstest_over_refusal'], r['harmbench_harm']
    near = any(abs(x0 - a) < 0.8 and abs(y0 - b) < 0.5 for a, b in placed)
    ax.scatter(x0, y0, s=40, color=COLOR[m], zorder=3)
    ax.annotate(LABEL[m], (x0, y0), xytext=(4, -9 if near else 2), textcoords='offset points', fontsize=7.5)
    placed.append((x0, y0))
if 'harmbench_harm' in base:
    ax.scatter(base['xstest_over_refusal'], base['harmbench_harm'], s=40, marker='s', color=COLOR['base'], zorder=3)
    ax.annotate('Base model', (base['xstest_over_refusal'], base['harmbench_harm']), xytext=(4, 2), textcoords='offset points', fontsize=7.5)
ax.set_xlabel('XSTest over-refusal % (lower better)')
ax.set_ylabel('HarmBench harmful % (lower better)')
ax.set_title('Safety vs over-refusal')
grid(ax, 'both')
ax = axes[1]
ife = GEN.groupby('method')['ifeval_prompt'].mean().reindex(gm)
y = np.arange(len(gm))[::-1]
ax.barh(y, ife, color=[COLOR[m] for m in gm], height=0.62, zorder=2)
if 'ifeval_prompt' in base:
    ax.axvline(base['ifeval_prompt'], color=INK, lw=1, ls='--')
    ax.text(base['ifeval_prompt'], len(gm) - 0.4, ' base', fontsize=7.5)
for yi, v in zip(y, ife):
    ax.text(v + 0.2, yi, f'{v:.1f}', va='center', fontsize=8)
ax.set_yticks(y, lab(gm))
ax.tick_params(axis='y', length=0)
ax.set_xlim(ife.min() - 3, ife.max() + 1.5)
ax.set_xlabel('IFEval prompt-level strict %')
ax.set_title('Instruction following')
grid(ax)
keep(fig, 'F9_generation')

## 7. Reward judge (seed 2)
A reward model scores two answers to the same prompt; the higher score wins (ties count half).
- **Head-to-head**: final answers of two methods on the same test prompts, all 4 orders. Above 50% = the first method's answers are preferred.
- **Final vs just-learned**: a method's final answer against its own answer right after that behavior was trained. 50% = no forgetting in the answers, below 50% = answers got worse. Only a few runs have this, so treat it as supporting evidence.

The ± is a 95% interval that treats prompts as independent, so it is a little optimistic.

In [ ]:
rows = []
for f in glob.glob(str(RUNS / 'v2_o[1-4]_*_s[0-2]' / 'generation' / '**' / 'judges' / '*' / 'summary.csv'), recursive=True):
    parts = Path(f).parts
    run, comp = parts[parts.index('runs') + 1], Path(f).parent.name
    o, m, s = re.match(r'v2_o(\d)_(.+)_s(\d)$', run).groups()
    d = pd.read_csv(f).assign(run=run, method=m, order=int(o), seed=int(s))
    d['comparison'] = comp if comp == 'final_vs_post_stage' else 'vs_' + method_of(comp.replace('vs_', ''))
    rows.append(d)
JUDGE = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if len(JUDGE):
    JUDGE['points'] = JUDGE['candidate_wins'] + 0.5 * JUDGE['ties']
    t8 = JUDGE.groupby(['comparison', 'method', 'behavior']).agg(
        runs=('run', 'nunique'), orders=('order', lambda x: ','.join(map(str, sorted(set(x))))),
        prompts=('valid_examples', 'sum'), points=('points', 'sum'))
    p = t8['points'] / t8['prompts']
    t8['win %'] = 100 * p
    t8['± 95%'] = 196 * np.sqrt(p * (1 - p) / t8['prompts'])
    t8 = t8.drop(columns='points').reset_index()
    t8['comparison'] = np.where(t8['comparison'] == 'final_vs_post_stage', 'final vs just-learned',
                                t8['method'].map(LABEL) + ' vs ' + t8['comparison'].str[3:].map(LABEL))
    t8['method'] = t8['method'].map(LABEL)
    t8['behavior'] = t8['behavior'].map(BLABEL)
    show(t8.set_index(['comparison', 'method', 'behavior']), 'T8_reward_judge', 'Pooled win rates.', 1)

In [ ]:
# Figure 10: judge win rates
if len(JUDGE):
    h2h = t8[t8['comparison'] != 'final vs just-learned']
    fvl = t8[t8['comparison'] == 'final vs just-learned']
    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.7), width_ratios=[1, 1.3])
    for ax, t, key, title in [(axes[0], h2h, 'comparison', 'Head-to-head, final answers'),
                              (axes[1], fvl, 'method', 'Final vs just-learned (50% = no forgetting)')]:
        groups = list(dict.fromkeys(t[key]))
        for j, b in enumerate(['Helpful', 'Quality']):
            s_ = t[t['behavior'] == b].set_index(key).reindex(groups)
            x = np.arange(len(groups)) + (j - 0.5) * 0.36
            ax.bar(x, s_['win %'] - 50, 0.34, bottom=50, color=BCOLOR['helpful' if b == 'Helpful' else 'quality'],
                   label=b, zorder=2)
            ax.errorbar(x, s_['win %'], yerr=s_['± 95%'], fmt='none', ecolor=INK, lw=0.9, capsize=2, zorder=3)
        ax.axhline(50, color=INK, lw=0.8)
        ax.set_xticks(range(len(groups)), [g.replace(' vs ', '\nvs ') for g in groups], fontsize=7.5)
        ax.set_ylabel('Win rate (%)')
        ax.set_title(title, fontsize=9)
        grid(ax, 'y')
    axes[0].legend(fontsize=7.5, loc='lower right')
    keep(fig, 'F10_reward_judge')

## 8. Hypotheses
Verdicts are worked out from the numbers above, so they update if the data changes. H1–H4 and H8 use the locked test; H5–H7 use validation.

In [ ]:
def pt(df, a, b, col='retention'):
    p = paired(df, a, b, col)
    return p, f"{p['diff']:+.2f} [{p['lo']:+.2f}, {p['hi']:+.2f}], {p['better']}/{p['cells']} cells"

nb = {b: TEST.loc[TEST['method'] == 'none', f'ret_{b}'].mean() for b in BEH}
p2, s2 = pt(TEST, 'mfr', 'random')
p3, s3 = pt(TEST, 'random_high', 'random')
p4, s4 = pt(TEST, 'mfr', 'lowest_margin')
p5, s5 = pt(VAL, 'mfr_balanced', 'mfr')
p6, s6 = pt(VAL, 'mfr_at_risk', 'mfr')
p8, s8 = pt(TEST, 'dapr_weak', 'lowest_margin')
p8n, _ = pt(TEST, 'dapr_weak', 'lowest_margin', 'new_task')
dist = R.groupby('method')['distinct'].mean()
H = [
    ('H1', 'Sequential DPO forgets earlier behaviors, Safety most',
     f"No replay {TEST.loc[TEST['method'] == 'none', 'retention'].mean():.2f}; Helpful {nb['helpful']:.2f}, "
     f"Safety {nb['safe']:.2f}, Quality {nb['quality']:.2f}",
     'holds' if min(nb, key=nb.get) == 'safe' else 'partly'),
    ('H2', 'MFR forgets less than Random 10%', s2, 'holds' if p2['lo'] > 0 else 'not shown'),
    ('H3', 'More random replay helps less than better selection',
     f'Random 14.3% − Random {s3}; MFR − Random {p2["diff"]:+.2f}', 'holds' if p3['diff'] < p2['diff'] else 'does not hold'),
    ('H4', 'MFR forgets less than Lowest Margin', s4, 'holds' if p4['lo'] > 0 else 'does not hold'),
    ('H5', 'MFR gives one behavior too many slots (Balanced MFR helps)', s5,
     'supported' if p5['lo'] > 0 else 'not supported'),
    ('H6', 'MFR misses pairs that are already wrong (At-Risk MFR helps)', s6,
     'supported' if p6['lo'] > 0 else 'not supported'),
    ('H7', 'MFR keeps replaying the same few pairs',
     f"distinct pairs per stage: MFR {dist.get('mfr', np.nan):.0f}, Lowest Margin {dist.get('lowest_margin', np.nan):.0f}, "
     f"Random {dist.get('random', np.nan):.0f}",
     'not supported' if dist.get('mfr', 0) >= dist.get('lowest_margin', 0) else 'supported'),
    ('H8', 'DAPR forgets less than Lowest Margin and loses < 2 points on the new behavior',
     f'retention {s8}; new task {p8n["diff"]:+.2f}', 'holds' if p8['lo'] > 0 and p8n['diff'] > -2 else 'does not hold'),
]
show(pd.DataFrame(H, columns=['', 'Hypothesis', 'Evidence', 'Verdict']).set_index(''), 'T9_hypotheses')

## 9. Read the outputs yourself
Small helpers that print real examples as cards. Change the arguments to look at other runs, behaviors or methods. Long texts are cut and can be expanded.

In [ ]:
CHARS = 1200
RESP_KEYS = ['response', 'output', 'generation', 'completion', 'answer', 'text']

def block(title, text, color=INK2):
    t = str(text if text is not None else '')
    more = '' if len(t) <= CHARS else (f'… <details style="display:inline"><summary style="cursor:pointer;color:#2a78d6">'
                                       f'show all ({len(t)} chars)</summary>{html.escape(t)}</details>')
    return (f'<div style="margin:6px 0"><div style="font-weight:600;font-size:12px;color:{color}">{html.escape(title)}</div>'
            f'<div style="white-space:pre-wrap;font-size:12.5px;line-height:1.35;border-left:3px solid {color};'
            f'padding:3px 8px;background:#fafaf8">{html.escape(t[:CHARS])}{more}</div></div>')

def card(header, blocks):
    display(HTML(f'<div style="border:1px solid #e5e4e0;border-radius:6px;padding:8px 12px;margin:10px 0">'
                 f'<div style="font-size:13px;font-weight:700;margin-bottom:4px">{html.escape(header)}</div>{"".join(blocks)}</div>'))

def generations(run, sub, fname):
    # id -> row (with 'prompt' and 'response') for one generation file of a run
    p = RUNS / run / 'generation' / sub / fname
    if not p.exists():
        return {}
    out = {}
    for i, r in enumerate(read_jsonl(p)):
        key = next((k for k in RESP_KEYS if k in r), None)
        r['response'] = r.get(key, '') if key else ''
        out[r.get('id', i)] = r
    return out

def prompt_of(pid, row=None):
    return (row or {}).get('prompt') or TEXT.get(pid, {}).get('prompt', '')

def judged_runs():
    return sorted({p.parts[p.parts.index('runs') + 1] for p in RUNS.glob('*/generation/**/judges/*/summary.csv')})
print('runs with judge files:', ', '.join(judged_runs()))

**Head-to-head.** The prompts where the judge preferred DAPR-weak's answer most over Lowest Margin's (`side='wins'`), or the reverse (`side='losses'`).

In [ ]:
def head_to_head(order=1, behavior='helpful', a='dapr_weak', b='lowest_margin', seed=2, n=3, side='wins'):
    ra, rb = f'v2_o{order}_{a}_s{seed}', f'v2_o{order}_{b}_s{seed}'
    f = RUNS / ra / 'generation' / 'final_eval' / 'judges' / f'vs_{rb}' / f'{behavior}.csv'
    if not f.exists():
        print('no judge file:', f)
        return
    j = pd.read_csv(f)
    j['gap'] = j['candidate_reward'] - j['baseline_reward']
    j = j.sort_values('gap', ascending=(side != 'wins')).head(n)
    A = generations(ra, 'final_eval', f'{behavior}_test.jsonl')
    B = generations(rb, 'final_eval', f'{behavior}_test.jsonl')
    for _, r in j.iterrows():
        card(f'{BLABEL[behavior]} · {r["id"]} · order {order} · reward {LABEL[a]} {r["candidate_reward"]:.2f} vs '
             f'{LABEL[b]} {r["baseline_reward"]:.2f}',
             [block('Prompt', prompt_of(r['id'], A.get(r['id']))),
              block(LABEL[a], A.get(r['id'], {}).get('response', '(missing)'), COLOR[a]),
              block(LABEL[b], B.get(r['id'], {}).get('response', '(missing)'), COLOR[b])])

head_to_head(order=1, behavior='helpful', side='wins')
head_to_head(order=1, behavior='helpful', side='losses', n=2)

**Forgetting in the answers.** A run's final answer next to its own answer right after that behavior was trained, where the final answer lost the most.

In [ ]:
def forgetting_examples(run='v2_o1_dapr_weak_s2', behavior='helpful', n=3):
    f = RUNS / run / 'generation' / 'judges' / 'final_vs_post_stage' / f'{behavior}.csv'
    if not f.exists():
        print('no file:', f, '\nruns with judges:', judged_runs())
        return
    j = pd.read_csv(f)
    j['gap'] = j['candidate_reward'] - j['baseline_reward']
    j = j.sort_values('gap').head(n)
    post = sorted((RUNS / run / 'generation').glob(f'post_stage*{behavior}*'))
    P = generations(run, post[0].name, f'{behavior}_test.jsonl') if post else {}
    F = generations(run, 'final_eval', f'{behavior}_test.jsonl')
    m = method_of(run)
    for _, r in j.iterrows():
        card(f'{LABEL.get(m, m)} · {run} · {r["id"]} · reward final {r["candidate_reward"]:.2f} vs just-learned {r["baseline_reward"]:.2f}',
             [block('Prompt', prompt_of(r['id'], F.get(r['id']))),
              block('Right after learning ' + BLABEL[behavior], P.get(r['id'], {}).get('response', '(missing)'), '#1baf7a'),
              block('End of training', F.get(r['id'], {}).get('response', '(missing)'), '#e34948')])

forgetting_examples('v2_o1_lowest_margin_s2', 'helpful', n=2)
forgetting_examples('v2_o1_dapr_weak_s2', 'helpful', n=2)

**Safety.** Answers WildGuard marked harmful on HarmBench, or safe XSTest requests the model refused.

In [ ]:
def to_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(['true', '1', 'yes', 'harmful', 'refusal', 'unsafe'])

def find_col(df, *keys, exclude=()):
    return next((c for c in df.columns if all(k in c.lower() for k in keys) and not any(e in c.lower() for e in exclude)), None)

def safety_examples(run='v2_o1_none_s0', suite='harmbench', kind='harmful', n=3):
    wg = RUNS / run / 'generation' / 'final_eval' / 'safety' / f'wildguard_{suite}.csv'
    if not wg.exists():
        print('no file:', wg)
        return
    d = pd.read_csv(wg)
    gen = list(generations(run, 'final_eval', f'{suite}.jsonl').values())
    if kind == 'harmful':
        col = find_col(d, 'response', 'harm') or find_col(d, 'harm', exclude=('prompt',))
        mask = to_bool(d[col]) if col else None
    else:
        col = find_col(d, 'refus', exclude=('prompt',))
        safe_col = find_col(d, 'prompt', 'harm')
        mask = to_bool(d[col]) & ~to_bool(d[safe_col]) if col and safe_col else (to_bool(d[col]) if col else None)
    if mask is None:
        print('could not find the label column; columns are:', list(d.columns))
        return
    hits = d[mask].head(n)
    print(f'{run} · {suite}: {int(mask.sum())} of {len(d)} answers flagged ({kind})')
    for i, r in hits.iterrows():
        g = next((x for x in gen if 'id' in d.columns and x.get('id') == r.get('id')), gen[i] if i < len(gen) else {})
        card(f'{LABEL.get(method_of(run), run)} · {run} · {suite} · {kind}',
             [block('Prompt', r.get('prompt') or g.get('prompt', '')),
              block('Answer', r.get('response') or g.get('response', ''), '#e34948')])

safety_examples('v2_o1_none_s0', 'harmbench', 'harmful', n=2)
safety_examples('v2_o1_lowest_margin_s0', 'xstest', 'over_refusal', n=2)

**Preference pairs.** A pair from the shared core (forgotten by every midterm method), with each method's margin right after learning and at the end.

In [ ]:
def pair_card(order, seed, behavior, pid):
    r = TEXT.get(pid, {})
    g = PAIRS_DF[(PAIRS_DF['order'] == order) & (PAIRS_DF['seed'] == seed) & (PAIRS_DF['id'] == pid)].set_index('method')
    rows = ''.join(
        f'<tr><td style="padding:2px 10px">{LABEL[m]}</td><td style="padding:2px 10px;text-align:right">{g.loc[m, "learned"]:+.4f}</td>'
        f'<td style="padding:2px 10px;text-align:right">{g.loc[m, "final"]:+.4f}</td>'
        f'<td style="padding:2px 10px;color:{"#e34948" if g.loc[m, "forgotten"] else "#1baf7a"}">'
        f'{"forgotten" if g.loc[m, "forgotten"] else ("kept" if g.loc[m, "learned"] > 0 else "never learned")}</td></tr>'
        for m in PAIR_METHODS if m in g.index)
    table = ('<table style="font-size:12px;margin:6px 0"><tr><th style="text-align:left;padding:2px 10px">Method</th>'
             '<th style="padding:2px 10px">Margin after learning</th><th style="padding:2px 10px">Margin at end</th><th></th></tr>'
             + rows + '</table>')
    card(f'{BLABEL[behavior]} · {pid} · order {order}, seed {seed} · chosen − rejected = {gap.get(pid, np.nan):+.0f} tokens',
         [block('Prompt', r.get('prompt', '')), block('Chosen', r.get('chosen', ''), '#1baf7a'),
          block('Rejected', r.get('rejected', ''), '#e34948'), table])

for (o, s, b, pid) in list(core_ids[:3]):
    pair_card(o, s, b, pid)

## 10. Save

In [ ]:
for name, df in TABLES.items():
    df.to_csv(OUT / f'{name}.csv')
for name, fig in FIGS.items():
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', pad_inches=0.04)
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight', pad_inches=0.04)
print(f'saved {len(TABLES)} tables and {len(FIGS)} figures to {OUT}')